In [1]:
import xarray as xr
import earthaccess
import boto3
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import warnings
from IPython.display import display, Markdown
import pandas as pd
import geopandas as gpd
import rasterio
import datetime
import pyarrow as pa
import pyarrow.parquet as pq
import os

warnings.filterwarnings('ignore')
%matplotlib inline

In [2]:
fires = pd.read_parquet("s3://maap-ops-workspace/shared/zbecker/TESS_fire_spread/sigdeltas_Tess.parq")
subset_fires = gpd.read_parquet("s3://maap-ops-workspace/shared/zbecker/YANG/large_feds_faf_double_matched.parq")
subset_fires = subset_fires.to_crs(4326)

subset_fires["centroid"] = subset_fires.to_crs(4326).centroid
fires["UfireID"] = fires.mergeid.astype("int").astype("str") + "_" + fires.year.astype("str")
subset_fires["UfireID"] = subset_fires.mergeid.astype("str") + "_" + subset_fires.year.astype("str")
subset_fires["polygon"] = subset_fires.geometry
fires = fires[fires.UfireID.isin(subset_fires[subset_fires.intersectsMTBS == True].UfireID)]
fname = pd.read_csv("s3://maap-ops-workspace/shared/zbecker/Eli_MTBS_vs_FEDS/v6_output.csv")

fires = fires.merge(subset_fires[['UfireID', 'centroid', 'polygon']], on = 'UfireID' )
fires = gpd.GeoDataFrame(fires, geometry = 'polygon')

def get_st_sp_fire(df, days_after = 7):
    df.loc[:, "start_time"] = df.t.min()
    df.loc[:, "end_time"] = df.t.max()
    df.loc[:, "end_time_plus"] = df.t.astype("datetime64[ns]").max()  + datetime.timedelta(days = days_after)
    df = df.loc[df.t == df.t.max(), :]
    return(df)
    
fires_sm = fires.groupby("UfireID").apply(get_st_sp_fire).reset_index(drop = True)
fires_sm["stable_index"] = fires_sm.index

In [37]:
precip = pd.read_parquet(os.path.abspath("IMERG/half_hourly_IMERG_precip"))

In [ ]:
precip = precip.merge(fires_sm[["UfireID",	"centroid"]], on = "UfireID")

precip.loc[:, "lon"] = precip.centroid.apply(lambda p: p.x)

In [58]:
precip.loc[:, "offset_hour"] = (precip.lon/15)

precip.loc[:, "time_lst"] = precip.time_utc.astype("datetime64[ns]") + pd.to_timedelta(precip["offset_hour"], unit='h') #lst_to_utc_offset_hours =  lon/ 15.0

pm_mask = (precip.time_lst.dt.hour > 6) & (precip.time_lst.dt.hour <= 18) ## correcting to PM 13:30 overpass
am_mask = (precip.time_lst.dt.hour <= 6) | ((precip.time_lst.dt.hour >= 18))## correcting to AM 1:30 overpass. This is actually an exact  number not a range bc we calcuated it for the extraction. 

precip.loc[pm_mask, "t"] = precip.loc[pm_mask, "time_lst"].dt.normalize() + pd.Timedelta(hours=12)
precip.loc[am_mask, "t"] = precip.loc[am_mask, "time_lst"].dt.normalize() + pd.Timedelta(hours=0)

